# XAUUSD RL — Ensemble of the Last N Sliding Folds (Portfolio)

**Purpose**: combine the most recent `N` sliding walk-forward fold models and
reproduce the `test_analysis.ipynb` visualizations.

**Ensemble method = PORTFOLIO (equity blend).** Each member runs as its own book
(its own coherent positions), and we blend the equity curves with equal capital.
This is the correct ensemble for a *sequential, path-dependent* trading policy.
A "naive action-vote" ensemble (averaging per-bar action probabilities) is also
computed and shown **only for contrast** — averaging actions produces incoherent
compromise trades and underperforms, so it is not used for the analysis.

- Default eval window = the **last fold's test window** (strictly OOS for every
  member). Set `STRICT_OOS=False` for the wider last-`N` span (context only).
- Charts render inline and export to `outputs/notebook_ensemble/`.

## 0 · Setup (paths + Windows DLL fix)

In [20]:
import json, os, sys, site
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.py").exists(), f"Could not find project root from {Path.cwd()}"
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Project root:", ROOT)

if os.name == "nt":
    _sp = list(site.getsitepackages())
    try:
        _sp.append(site.getusersitepackages())
    except Exception:
        pass
    _prepend = [str(Path(sys.executable).parent)]
    for _s in _sp:
        _tl = Path(_s) / "torch" / "lib"
        if _tl.exists():
            _prepend.append(str(_tl))
        for _p in (Path(_s) / "nvidia").glob("*/lib"):
            if _p.exists():
                _prepend.append(str(_p))
    os.environ["PATH"] = os.pathsep.join(_prepend) + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        for _d in _prepend:
            try:
                os.add_dll_directory(_d)
            except Exception:
                pass

import numpy as np
import pandas as pd
OUT = ROOT / "outputs" / "notebook_ensemble"
OUT.mkdir(parents=True, exist_ok=True)
print("Charts ->", OUT)

Project root: f:\CodeTrading\Code_trading_2026\34_RL_XAUUSD
Charts -> f:\CodeTrading\Code_trading_2026\34_RL_XAUUSD\outputs\notebook_ensemble


## 1 · Pick the last N folds

In [21]:
N_FOLDS = 5        # ensemble size (most recent folds)
STRICT_OOS = True  # True : eval on the last fold's test window (clean OOS for every member)
                   # False: eval on the whole last-N test span (more data; context only)

SLIDING = ROOT / "models" / "sliding"
fold_dirs = [d for d in sorted(SLIDING.glob("fold_*"), key=lambda p: int(p.name.split("_")[1]))
             if (d / "run_info.json").exists()]
assert fold_dirs, f"No completed sliding folds under {SLIDING}. Run train_ppo.py (sliding) first."
last_dirs = fold_dirs[-N_FOLDS:]
print(f"Found {len(fold_dirs)} completed folds; ensembling the last {len(last_dirs)}:")
for d in last_dirs:
    print("  ", d.name)

summary_path = ROOT / "models" / "sliding_walk_forward_summary.csv"
summary = pd.read_csv(summary_path) if summary_path.exists() else None
if summary is not None:
    cols = [c for c in ["fold", "test_start", "test_end", "test_return_pct",
                        "test_profit_factor", "test_sharpe", "test_n_trades"] if c in summary.columns]
    print("\nPer-fold TEST metrics (last N):")
    print(summary[cols].tail(N_FOLDS).to_string(index=False))

Found 35 completed folds; ensembling the last 5:
   fold_31
   fold_32
   fold_33
   fold_34
   fold_35

Per-fold TEST metrics (last N):
 fold test_start   test_end  test_return_pct  test_profit_factor  test_sharpe  test_n_trades
   31 2023-12-01 2024-05-18        12.713465            1.297093     2.089971          170.0
   32 2024-05-30 2024-11-19        -3.840273            0.914404    -0.790270          225.0
   33 2024-12-02 2025-05-19         5.507777            1.098825     0.944220          203.0
   34 2025-05-30 2025-11-19        10.606759            1.203543     1.658446          195.0
   35 2025-12-02 2026-05-19        15.851003            1.175164     2.048103          317.0


## 2 · Load data & feature frame

In [22]:
from config import CFG
from data_loader import load_mt_ohlcv_csv, resample_ohlcv
from features import prepare_feature_frame

print("Loading M1 ...  (full", CFG.csv_path.name, "- this can take a while)")
m1 = load_mt_ohlcv_csv(
    CFG.csv_path, time_col=CFG.time_col, source_tz=CFG.source_tz,
    timestamp_is_bar_open=CFG.timestamp_is_bar_open, bar_duration=CFG.pandas_execution_tf,
    start_date=CFG.start_date, end_date=CFG.end_date, max_days_for_demo=CFG.max_days_for_demo,
)
decision = resample_ohlcv(m1, CFG.pandas_tf)
feat, feature_cols = prepare_feature_frame(
    decision, warmup_bars=CFG.warmup_bars, atr_period=CFG.atr_period, rsi_period=CFG.rsi_period)

def _m1_slice(df):
    return m1.loc[(m1.index > df.index.min()) & (m1.index <= df.index.max())]

print(f"feat: {len(feat):,} bars  {feat.index.min().date()} -> {feat.index.max().date()}  ({len(feature_cols)} features)")

Loading M1 ...  (full XAUUSD_1 Min_Bid_2003.05.05_2026.05.31.csv - this can take a while)
feat: 138,738 bars  2003-05-19 -> 2026-05-30  (25 features)


## 3 · Load the last-N fold models + their VecNormalize

Each member prefers its **best (eligible) checkpoint** and the normalisation
snapshot it was selected under; it falls back to the final checkpoint otherwise.

In [23]:
import pickle
from stable_baselines3 import PPO

def load_member(fold_dir):
    ri = json.loads((fold_dir / "run_info.json").read_text())
    mp = ri.get("best_model_path") or ri["model_path"]
    vp = ri.get("best_model_vecnorm_path") or ri["vecnorm_path"]
    if not Path(mp).exists():
        mp, vp = ri["model_path"], ri["vecnorm_path"]
    model = PPO.load(str(mp), device="cpu")
    model.policy.set_training_mode(False)
    with open(vp, "rb") as f:
        vn = pickle.load(f)
    return fold_dir.name, model, vn

members = [load_member(d) for d in last_dirs]
labels = [m[0] for m in members]
models = [m[1] for m in members]
vnorms = [m[2] for m in members]
print("Ensemble members:", labels)
print("Obs dims:", [m.observation_space.shape for m in models])

Ensemble members: ['fold_31', 'fold_32', 'fold_33', 'fold_34', 'fold_35']
Obs dims: [(31,), (31,), (31,), (31,), (31,)]


## 4 · Per-member rollout helper (and the naive action-vote agent)

Each member is rolled out on its **own** env (its own positions), normalising the
observation with its **own** VecNormalize. The portfolio (section 6) then blends
the resulting equity curves. `EnsembleAgent` is also used to build the *naive*
action-vote ensemble shown for contrast.

In [24]:
import torch
from env_bracket import BracketTradingEnv

class EnsembleAgent:
    """Averages members' per-bar action probabilities (naive action-vote). With a
    single member it is just that member's deterministic policy."""
    def __init__(self, models, vnorms):
        self.models = models
        self.vnorms = vnorms
    def predict(self, raw_obs):
        acc = None
        for model, vn in zip(self.models, self.vnorms):
            o = vn.normalize_obs(np.asarray(raw_obs, dtype=np.float64).reshape(1, -1)).astype(np.float32)
            with torch.no_grad():
                dist = model.policy.get_distribution(torch.as_tensor(o, device=model.policy.device))
            probs = [torch.softmax(c.logits, dim=-1) for c in dist.distribution]
            acc = [p.clone() for p in probs] if acc is None else [a + p for a, p in zip(acc, probs)]
        return np.array([int(p.argmax(dim=-1).item()) for p in acc], dtype=int)

def make_raw_env(decision_df, m1_df):
    return BracketTradingEnv(
        decision_df, m1_df, feature_cols,
        sl_atr_multipliers=CFG.sl_atr_multipliers, tp_r_multipliers=CFG.tp_r_multipliers,
        initial_equity=CFG.initial_equity, risk_fraction=CFG.risk_fraction,
        spread_price=CFG.spread_price, slippage_price=CFG.slippage_price,
        commission_per_trade=CFG.commission_per_trade, holding_penalty=CFG.holding_penalty,
        reward_mtm_weight=CFG.reward_mtm_weight, randomize_start=False)

def rollout(agent, decision_df, m1_df):
    env = make_raw_env(decision_df, m1_df)
    obs, _ = env.reset()
    done = False
    while not done:
        obs, _r, term, trunc, _i = env.step(agent.predict(obs))
        done = term or trunc
    return env.equity_curve(), env.trade_log()

## 5 · Evaluation window

`STRICT_OOS=True` uses the **last fold's test window** (out-of-sample for every
member). `False` uses the full last-`N` test span (more data, context only).

In [25]:
if summary is not None:
    last5 = summary.tail(N_FOLDS)
    if STRICT_OOS:
        w_start = pd.Timestamp(last5.iloc[-1]["test_start"]); w_end = pd.Timestamp(last5.iloc[-1]["test_end"])
    else:
        w_start = pd.Timestamp(last5.iloc[0]["test_start"]);  w_end = pd.Timestamp(last5.iloc[-1]["test_end"])
    tz = feat.index.tz
    w_start = w_start.tz_localize(tz) if w_start.tzinfo is None else w_start.tz_convert(tz)
    w_end = w_end.tz_localize(tz) if w_end.tzinfo is None else w_end.tz_convert(tz)
    eval_feat = feat.loc[w_start:w_end]
else:
    eval_feat = feat.loc[feat.index.max() - pd.DateOffset(months=6):]
eval_m1 = _m1_slice(eval_feat)
print(f"Eval window: {eval_feat.index.min().date()} -> {eval_feat.index.max().date()}  ({len(eval_feat):,} bars)")
print("Mode:", "STRICT OOS (last fold test)" if STRICT_OOS else "WIDE last-N span (context only)")

Eval window: 2025-12-02 -> 2026-05-19  (2,685 bars)
Mode: STRICT OOS (last fold test)


## 6 · Run each member, then build the PORTFOLIO ensemble

**Portfolio** = equal-capital blend of the members' equity curves (each member
keeps its own positions; we average the books). The **naive action-vote** is also
run for contrast. Downstream cells use the portfolio as `ens_eq` / `ens_trades`.

In [26]:
# Each member as its own book.
member_eq, member_trades = {}, {}
for lbl, model, vn in zip(labels, models, vnorms):
    print(f"Rolling out {lbl} ...")
    e, t = rollout(EnsembleAgent([model], [vn]), eval_feat, eval_m1)
    member_eq[lbl], member_trades[lbl] = e, t

# Portfolio = equal-capital blend = average of the member equity curves
# (aligned on the shared decision-bar index). Pooled trades describe the book.
eq_mat = pd.concat([member_eq[l]["equity"].rename(l) for l in labels], axis=1).ffill()
portfolio_eq = eq_mat.mean(axis=1).to_frame("equity")
portfolio_trades = pd.concat([member_trades[l] for l in labels], ignore_index=True)

# Naive action-vote ensemble (for contrast only).
print("Rolling out NAIVE action-vote ensemble ...")
naive_eq, naive_trades = rollout(EnsembleAgent(models, vnorms), eval_feat, eval_m1)

from evaluate import full_report
def _rep(eq, tr):
    return full_report(eq, tr, initial_equity=CFG.initial_equity,
                       periods_per_year=CFG.periods_per_year)["value"].to_dict()

portfolio_rep = _rep(portfolio_eq, portfolio_trades)
naive_rep = _rep(naive_eq, naive_trades)

# Downstream cells operate on the PORTFOLIO ensemble.
ens_eq, ens_trades, ens_rep = portfolio_eq, portfolio_trades, portfolio_rep

print(f"\nPORTFOLIO ensemble : return={portfolio_rep.get('total_return_pct'):+.1f}%  "
      f"PF={portfolio_rep.get('profit_factor'):.2f}  Sharpe={portfolio_rep.get('sharpe_like'):+.2f}  "
      f"MaxDD={portfolio_rep.get('max_drawdown_pct'):+.1f}%")
print(f"Naive action-vote  : return={naive_rep.get('total_return_pct'):+.1f}%  "
      f"PF={naive_rep.get('profit_factor'):.2f}  Sharpe={naive_rep.get('sharpe_like'):+.2f}  (contrast only)")

Rolling out fold_31 ...
Rolling out fold_32 ...
Rolling out fold_33 ...
Rolling out fold_34 ...
Rolling out fold_35 ...
Rolling out NAIVE action-vote ensemble ...

PORTFOLIO ensemble : return=+11.9%  PF=1.16  Sharpe=+2.79  MaxDD=-7.6%
Naive action-vote  : return=-4.5%  PF=0.92  Sharpe=-0.70  (contrast only)


## 7 · Performance table — portfolio vs naive vote vs members

In [27]:
from visualize import plot_metrics_table, save_html
metrics = {"Portfolio": portfolio_rep, "Naive vote": naive_rep}
for lbl in labels:
    metrics[lbl] = _rep(member_eq[lbl], member_trades[lbl])
fig = plot_metrics_table(metrics, title=f"Portfolio vs naive vs members  |  {eval_feat.index.min().date()} -> {eval_feat.index.max().date()}")
save_html(fig, OUT / "00_metrics_table.html"); fig.show()

## 8 · Equity curves — members (coloured) vs portfolio (black)

In [28]:
import plotly.graph_objects as go

# distinct colour per member; portfolio total in black; naive vote dashed grey
COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
          "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]

fig = go.Figure()
for k, lbl in enumerate(labels):
    eq = member_eq[lbl]["equity"].astype(float)
    fig.add_trace(go.Scatter(
        x=eq.index, y=eq / eq.iloc[0] * 100, mode="lines", name=lbl,
        line=dict(color=COLORS[k % len(COLORS)], width=1.6), opacity=0.9))

ne = naive_eq["equity"].astype(float)
fig.add_trace(go.Scatter(
    x=ne.index, y=ne / ne.iloc[0] * 100, mode="lines", name="Naive action-vote",
    line=dict(color="#9e9e9e", width=1.5, dash="dash")))

pe = portfolio_eq["equity"].astype(float)
fig.add_trace(go.Scatter(
    x=pe.index, y=pe / pe.iloc[0] * 100, mode="lines", name="PORTFOLIO (total)",
    line=dict(color="black", width=3.4)))

fig.add_hline(y=100, line_dash="dot", line_color="gray", opacity=0.4)
fig.update_layout(
    title="Ensemble equity - members (coloured) vs PORTFOLIO (black) vs naive vote (dashed)",
    template="plotly_white", height=560, hovermode="x unified",
    yaxis_title="Equity (start = 100)", xaxis_title="Date",
    legend=dict(orientation="h", y=1.04, x=0))
save_html(fig, OUT / "01_equity_comparison.html"); fig.show()

In [29]:
from visualize import plot_equity_and_drawdown
fig = plot_equity_and_drawdown(ens_eq, title="PORTFOLIO ensemble - Equity & Drawdown")
save_html(fig, OUT / "02_portfolio_equity_drawdown.html"); fig.show()

## 9 · Trade analysis (portfolio book — pooled member trades)

In [30]:
from visualize import plot_r_distribution
fig = plot_r_distribution(ens_trades, title="Portfolio - Realized R-multiple distribution")
save_html(fig, OUT / "03_r_distribution.html"); fig.show()

In [31]:
from visualize import plot_monthly_pnl
fig = plot_monthly_pnl(ens_trades, title="Portfolio - Monthly P&L (R-multiples)")
save_html(fig, OUT / "04_monthly_pnl.html"); fig.show()

f:\CodeTrading\Code_trading_2026\34_RL_XAUUSD\visualize.py:375: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  tr["month"] = pd.to_datetime(tr["exit_time"]).dt.to_period("M").astype(str)


In [32]:
from visualize import plot_exit_breakdown
fig = plot_exit_breakdown(ens_trades, title="Portfolio - Exit reason breakdown")
save_html(fig, OUT / "05_exit_breakdown.html"); fig.show()

## 10 · Agent behaviour (portfolio book)

In [33]:
from visualize import plot_bracket_usage
fig = plot_bracket_usage(ens_trades, sl_atr_multipliers=CFG.sl_atr_multipliers,
                         tp_r_multipliers=CFG.tp_r_multipliers,
                         title="Portfolio - bracket selection (SL x TP)")
save_html(fig, OUT / "06_bracket_heatmap.html"); fig.show()

In [34]:
from visualize import plot_entry_timing
fig = plot_entry_timing(ens_trades, title="Portfolio - Entry timing (hour x weekday)")
save_html(fig, OUT / "07_entry_timing.html"); fig.show()

## 11 · Rolling Sharpe (portfolio)

In [35]:
from visualize import plot_rolling_metrics
window = max(20, len(ens_eq) // 15)
fig = plot_rolling_metrics(ens_eq, window=window, periods_per_year=CFG.periods_per_year,
                           title=f"Portfolio - Equity + rolling Sharpe (window={window} bars)")
save_html(fig, OUT / "08_rolling_sharpe.html"); fig.show()

## 12 · Candles with portfolio trades

In [36]:
from visualize import plot_trades_on_chart
BARS = 300
fig = plot_trades_on_chart(eval_feat, ens_trades, n=BARS, title=f"Portfolio - last {BARS} bars with trades")
save_html(fig, OUT / "09_candles_trades.html"); fig.show()

## 13 · Feature correlation (eval period)

In [37]:
from visualize import plot_feature_correlation
fig = plot_feature_correlation(eval_feat, feature_cols, title="Eval period - feature correlation")
save_html(fig, OUT / "10_feature_correlation.html"); fig.show()

## 14 · Summary

In [38]:
print("=" * 72)
print(f"PORTFOLIO ensemble of {len(labels)} folds: {labels}")
print(f"Eval window: {eval_feat.index.min().date()} -> {eval_feat.index.max().date()}  "
      f"({'strict OOS' if STRICT_OOS else 'wide span - context'})")
print("=" * 72)
for name, m in metrics.items():
    print(f"{name:<14} ret={m.get('total_return_pct', float('nan')):+7.1f}%  "
          f"Sharpe={m.get('sharpe_like', float('nan')):+5.2f}  "
          f"MaxDD={m.get('max_drawdown_pct', float('nan')):+6.1f}%  "
          f"PF={m.get('profit_factor', float('nan')):.2f}  n={int(m.get('n_trades', 0))}")
print("\nPortfolio = equal-capital blend of member equity curves (the correct trading")
print("ensemble). 'Naive vote' averages per-bar actions and is shown only to illustrate")
print("why averaging actions underperforms a portfolio of the same models.")
print("\nCharts saved to", OUT)

PORTFOLIO ensemble of 5 folds: ['fold_31', 'fold_32', 'fold_33', 'fold_34', 'fold_35']
Eval window: 2025-12-02 -> 2026-05-19  (strict OOS)
Portfolio      ret=  +11.9%  Sharpe=+2.79  MaxDD=  -7.6%  PF=1.16  n=1340
Naive vote     ret=   -4.5%  Sharpe=-0.70  MaxDD= -13.0%  PF=0.92  n=197
fold_31        ret=   +5.7%  Sharpe=+0.95  MaxDD= -13.9%  PF=1.09  n=261
fold_32        ret=  +23.1%  Sharpe=+3.43  MaxDD=  -5.1%  PF=1.31  n=293
fold_33        ret=   +9.1%  Sharpe=+1.40  MaxDD= -11.1%  PF=1.13  n=246
fold_34        ret=   +6.8%  Sharpe=+1.06  MaxDD=  -8.4%  PF=1.10  n=221
fold_35        ret=  +14.7%  Sharpe=+1.91  MaxDD=  -5.9%  PF=1.16  n=319

Portfolio = equal-capital blend of member equity curves (the correct trading
ensemble). 'Naive vote' averages per-bar actions and is shown only to illustrate
why averaging actions underperforms a portfolio of the same models.

Charts saved to f:\CodeTrading\Code_trading_2026\34_RL_XAUUSD\outputs\notebook_ensemble
